# Etape 6 - Sensibilite au ratio de cout HF:BF

Analyse **post-hoc** (aucun entrainement). On recalcule deux couts sous plusieurs
ratios HF:BF, avec le modele resolution^2 **re-calibre** (C_HF=10 fixe, BF canonique
= 10/ratio) :
- **cout donnees** = n_hf x C_HF + n_bf x C_BF(ratio) ;
- **cout total** (pondere) = images HF vues x C_HF + images BF vues x C_BF(ratio).

Objectif : verifier que les **conclusions sont robustes** au choix (arbitraire) du
ratio, et justifier le ratio retenu.

> Prerequis : les JSON de resultats doivent contenir n_hf_pool / n_bf_pool /
> hf_images_seen / bf_images_seen (ajoutes a l'etape 6). Relancer baselines +
> strategies avant ce notebook.

## 1) Setup : Drive, module cout, chemins

In [ ]:
import os
import sys
import json
import importlib

import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    RESULTS_ROOT = '/content/drive/MyDrive/UTBM_PF22/results'
    SRC = '/content/drive/MyDrive/UTBM_PF22/src'
except Exception:
    RESULTS_ROOT = '../results'
    SRC = '../src'

for _p in [SRC, '../src', 'src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import cost
importlib.reload(cost)
from cost import unit_cost_ratio

OUT_DIR = os.path.join(RESULTS_ROOT, 'comparison')
os.makedirs(OUT_DIR, exist_ok=True)
print('Resultats   :', RESULTS_ROOT)
print('Module cout :', os.path.dirname(cost.__file__))
print('Sortie      :', OUT_DIR)

## 2) Configuration : ratios, modeles, datasets

In [ ]:
RATIOS = [2, 5, 10, 20, 50]
HF_SIZE = 224

# Couleurs coherentes avec les notebooks 11/12
MODELS = [
    ('BL 1(HF)',          'results_baseline_HF.json',                    '#4C72B0'),
    ('BL 2(BF)',          'results_baseline_BF.json',                    '#55A868'),
    ('BL 3(MIXTE)',       'results_baseline_MIXTE.json',                 '#C44E52'),
    ('Strat 1(BF->HF)',   'results_strategy1_transfer_learning.json',    '#8172B2'),
    ('Strat 2(CoTrain)',  'results_strategy2_cotraining_reweighting.json','#CCB974'),
    ('Strat 3(NoisyStud)','results_strategy3_noisy_student.json',        '#937860'),
]

DATASETS = [
    ('Animals-10', RESULTS_ROOT),
    ('Imagewoof',  os.path.join(RESULTS_ROOT, 'Imagewoof')),
]

REQUIRED = ['accuracy_HF', 'n_hf_pool', 'n_bf_pool', 'hf_images_seen', 'bf_images_seen']
print('Ratios testes :', RATIOS)

## 3) Chargement des resultats (avec decomposition par domaine)

In [ ]:
data = {}   # data[dataset][model] = {...}
for ds_name, res_dir in DATASETS:
    data[ds_name] = {}
    for disp, fname, color in MODELS:
        path = os.path.join(res_dir, fname)
        if not os.path.exists(path):
            continue
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        if not all(k in d for k in REQUIRED):
            print(f"  [ignore] {ds_name} | {disp} : champs manquants "
                  f"(relancer apres l'etape 6) -> {path}")
            continue
        data[ds_name][disp] = {
            'accuracy_HF': float(d['accuracy_HF']),
            'n_hf_pool': float(d['n_hf_pool']),
            'n_bf_pool': float(d['n_bf_pool']),
            'hf_images_seen': float(d['hf_images_seen']),
            'bf_images_seen': float(d['bf_images_seen']),
            'color': color,
        }
    print(f"{ds_name}: {len(data[ds_name])} modeles charges -> {list(data[ds_name])}")


def costs_at_ratio(m, ratio):
    """Retourne (cout_donnees, cout_total) pour un modele a un ratio donne."""
    c_hf = unit_cost_ratio(None, ratio, HF_SIZE)   # = 10 (constant)
    c_bf = unit_cost_ratio(64, ratio, HF_SIZE)     # = 10 / ratio
    data_cost = m['n_hf_pool'] * c_hf + m['n_bf_pool'] * c_bf
    total_cost = m['hf_images_seen'] * c_hf + m['bf_images_seen'] * c_bf
    return data_cost, total_cost

## 4) Tableau des couts par ratio (donnees et total)

In [ ]:
for ds_name, _ in DATASETS:
    if not data[ds_name]:
        continue
    print(f"\n===== {ds_name} =====")
    header = 'Modele'.ljust(20) + 'AccHF'.rjust(7) + '  | ' + \
        '  '.join([f'R={r}'.rjust(9) for r in RATIOS])
    print('--- COUT DONNEES (CA) ---')
    print(header)
    for disp, m in data[ds_name].items():
        row = disp.ljust(20) + f"{m['accuracy_HF']:6.2f}" + '  | '
        row += '  '.join([f"{costs_at_ratio(m, r)[0]:9.0f}" for r in RATIOS])
        print(row)
    print('--- COUT TOTAL (CA) ---')
    print(header)
    for disp, m in data[ds_name].items():
        row = disp.ljust(20) + f"{m['accuracy_HF']:6.2f}" + '  | '
        row += '  '.join([f"{costs_at_ratio(m, r)[1]:9.0f}" for r in RATIOS])
        print(row)

## 5) Efficacite (precision HF par 1000 CA) en fonction du ratio
Une courbe par modele. Si l'ordre des courbes ne change pas quand le ratio varie,
la conclusion est **robuste** au choix du ratio.

In [ ]:
def efficiency(m, ratio, kind):
    dc, tc = costs_at_ratio(m, ratio)
    cost_val = dc if kind == 'data' else tc
    return m['accuracy_HF'] / (cost_val / 1000.0) if cost_val > 0 else float('nan')

for ds_name, _ in DATASETS:
    if not data[ds_name]:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    fig.suptitle(f"Sensibilite au ratio HF:BF - {ds_name}", fontsize=14)
    for ax, kind, title in [(axes[0], 'data', 'Cout DONNEES'),
                            (axes[1], 'total', 'Cout TOTAL (pondere)')]:
        for disp, m in data[ds_name].items():
            ys = [efficiency(m, r, kind) for r in RATIOS]
            ax.plot(RATIOS, ys, marker='o', color=m['color'], label=disp)
        ax.set_xscale('log')
        ax.set_xticks(RATIOS)
        ax.set_xticklabels([str(r) for r in RATIOS])
        ax.set_title(f"{title} : efficacite vs ratio")
        ax.set_xlabel('Ratio de cout HF:BF')
        ax.set_ylabel('Precision HF par 1000 CA')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    suffix = 'animals' if ds_name == 'Animals-10' else 'imagewoof'
    out = os.path.join(OUT_DIR, f'cost_ratio_sensitivity_{suffix}.png')
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
    print('Figure sauvee:', out)

## 6) Robustesse : meilleur modele (efficacite) par ratio
Si le meilleur modele reste le meme sur toute la plage de ratios, la conclusion
ne depend pas du choix arbitraire du ratio.

In [ ]:
summary = {}
for ds_name, _ in DATASETS:
    if not data[ds_name]:
        continue
    summary[ds_name] = {}
    print(f"\n===== {ds_name} =====")
    for kind, label in [('data', 'Cout DONNEES'), ('total', 'Cout TOTAL')]:
        best_per_ratio = {}
        print(f"--- {label} : meilleur (efficacite HF/CA) par ratio ---")
        for r in RATIOS:
            ranked = sorted(data[ds_name].items(),
                            key=lambda kv: efficiency(kv[1], r, kind), reverse=True)
            best = ranked[0][0]
            best_per_ratio[r] = best
            order = ' > '.join([d for d, _ in ranked])
            print(f"  ratio {r:>3} : meilleur = {best:<18} | {order}")
        unique_best = set(best_per_ratio.values())
        robust = (len(unique_best) == 1)
        summary[ds_name][kind] = {'best_per_ratio': best_per_ratio, 'robust': robust}
        verdict = ('ROBUSTE (meme gagnant partout)' if robust
                   else f'SENSIBLE au ratio (gagnants: {sorted(unique_best)})')
        print(f"  => {verdict}\n")

# Sauvegarde JSON de synthese (couts + efficacites par ratio + robustesse)
export = {}
for ds_name, _ in DATASETS:
    if not data[ds_name]:
        continue
    export[ds_name] = {'ratios': RATIOS, 'models': {}, 'robustness': summary.get(ds_name, {})}
    for disp, m in data[ds_name].items():
        export[ds_name]['models'][disp] = {
            'accuracy_HF': m['accuracy_HF'],
            'data_cost': {r: costs_at_ratio(m, r)[0] for r in RATIOS},
            'total_cost': {r: costs_at_ratio(m, r)[1] for r in RATIOS},
        }
json_path = os.path.join(OUT_DIR, 'cost_ratio_sensitivity.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(export, f, indent=2, ensure_ascii=False)
print('JSON synthese sauve:', json_path)

## 7) Notes / justification du ratio

- Si le **meilleur modele** et l'**ordre** des courbes sont stables sur 5:1 -> 50:1,
  la conclusion (quelle strategie domine) ne depend PAS du choix arbitraire du ratio :
  on peut conserver **10:1** sans perte de generalite, en le justifiant a la fois par
  la robustesse et par le modele resolution^2 (le BF canonique 64px coute ~ pixels/224^2).
- Si l'ordre change a un certain ratio, c'est le seuil a discuter dans le rapport
  (ex. au-dela de tel ratio, telle strategie devient la plus efficace).
- Figures : `cost_ratio_sensitivity_{animals,imagewoof}.png` + JSON synthese, dans
  `results/comparison/`.